# UrbanFlow - reusable Spark pipeline

Run on a standard CPU Colab runtime. Mount the personal Drive holding the completed Q1 run, then execute the same reusable modules. A fresh run ID protects the immutable published version. Source downloads and processing use VM disk; the final serving archive is copied to Drive and checksum-verified. Full curated export is separately orchestrated by `scripts/colab-curated.py` from a repository checkout. Do not stop the runtime before required outputs persist.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import UTC, datetime
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/UrbanFlow")
ORIGINAL_RUN = DRIVE / "runs/yellow-2025-q1-v1"
# This pinned archive is the exact reusable code used for the reported Q1 run.
BUNDLE = ORIGINAL_RUN / "executed-source.zip"
EXPECTED_SHA256 = "b9aeb8ea22e05f9b3d776c90dd3d4ac256fc6bb86f5da5f2554e8005d5b95aae"
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError("Source bundle checksum mismatch")
REPO = Path("/content/urbanflow-notebook-source").resolve()
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for member in archive.infolist():
        if not (REPO / member.filename).resolve().is_relative_to(REPO):
            raise ValueError("Unsafe archive member")
    archive.extractall(REPO)
# Explicit dependencies also support the archived package before geometry/benchmark extras.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(REPO / "pipelines"),
        "pyspark==3.5.9",
        "pandas==2.2.3",
        "pyarrow>=19,<24",
        "psutil>=6,<8",
        "pyshp>=2.3,<3",
        "pyproj>=3.6,<4",
    ],
    check=True,
)
sys.path.insert(0, str(REPO / "pipelines/src"))
RAW = Path("/content/urbanflow-data/raw")
CONFIG = json.loads((REPO / "configs/yellow-2025-q1.json").read_text())
print("Verified source:", EXPECTED_SHA256)

In [ ]:
from urbanflow_etl.ingestion.download import download_dataset

source_manifest = download_dataset(CONFIG, RAW)
INPUTS = [str(RAW / item["name"]) for item in CONFIG["files"] if item["kind"] == "parquet"]
print("Validated source inputs:", INPUTS)

In [ ]:
# Set driver memory before creating any Spark session. Restart runtime to change it later.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("UrbanFlow notebook")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
from urbanflow_etl.analytics.job import run
from urbanflow_etl.benchmarking.job import run as benchmark

RUN_ID = "yellow-2025-q1-reproduction-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = Path("/content/urbanflow-results")
try:
    manifest = run(
        spark,
        INPUTS,
        RAW / "taxi_zone_lookup.csv",
        RAW / "taxi_zones.zip",
        OUTPUT,
        "2025-01-01",
        "2025-03-31",
        RUN_ID,
    )
    experiments = benchmark(spark, INPUTS, OUTPUT / "serving")
    print("Validated:", RUN_ID, "benchmark experiments:", len(experiments))
finally:
    spark.stop()

In [ ]:
archive = Path(
    shutil.make_archive("/content/urbanflow-reproduction-serving", "zip", OUTPUT / "serving")
)
destination = DRIVE / "runs" / RUN_ID
destination.mkdir(parents=True, exist_ok=False)
target = destination / "serving.zip"
shutil.copyfile(archive, target)
with archive.open("rb") as stream:
    source_hash = hashlib.file_digest(stream, "sha256").hexdigest()
with target.open("rb") as stream:
    target_hash = hashlib.file_digest(stream, "sha256").hexdigest()
if source_hash != target_hash:
    raise ValueError("Drive serving archive checksum mismatch")
shutil.copyfile(OUTPUT / "serving/manifest.json", destination / "manifest.json")
shutil.copyfile(OUTPUT / "serving/benchmarks.jsonl", destination / "benchmarks.jsonl")
print("Persisted verified serving archive:", target, source_hash)

To publish, download/extract the verified archive locally and invoke `python -m urbanflow_etl.publishing.mongo --input <serving-directory>` with `URBANFLOW_MONGODB_URI` supplied securely. This notebook intentionally contains no database credentials. Raw and complete curated data stay outside MongoDB.